In [ ]:
!pip install kaggle -q
!mkdir -p ~/.kaggle

In [ ]:
!kaggle datasets download jessicali9530/celeba-dataset

In [ ]:
!mkdir -p ./data/celeba
!unzip -q celeba-dataset.zip -d ./data/celeba

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import torchvision
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import os

## Config

In [ ]:
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IM_SIZE     = 32
im_channels = 3

T           = 1000     
beta_start  = 1e-4
beta_end    = 0.02

batch_size  = 64        
epochs      = 50      
lr          = 2e-4
num_samples = 100_000  

print(f"Device: {device}")
print(f"T={T} | batch={batch_size} | epochs={epochs} | lr={lr}")

## Noise Scheduler

In [ ]:
class LinearNoiseScheduler:
    def __init__(self, num_timesteps, beta_start, beta_end):
        self.num_timesteps = num_timesteps
        self.betas         = torch.linspace(beta_start, beta_end, num_timesteps)
        self.alphas        = 1.0 - self.betas
        self.alpha_cum_prod              = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_cum_prod         = torch.sqrt(self.alpha_cum_prod)
        self.sqrt_one_minus_alpha_cum_prod = torch.sqrt(1.0 - self.alpha_cum_prod)

    def add_noise(self, original, noise, t):
        batch_size = original.shape[0]
        sqrt_acp  = self.sqrt_alpha_cum_prod.to(original.device)[t].reshape(batch_size)
        sqrt_1macp = self.sqrt_one_minus_alpha_cum_prod.to(original.device)[t].reshape(batch_size)
        for _ in range(len(original.shape) - 1):
            sqrt_acp   = sqrt_acp.unsqueeze(-1)
            sqrt_1macp = sqrt_1macp.unsqueeze(-1)
        return sqrt_acp * original + sqrt_1macp * noise

    def sample_prev_timestep(self, xt, noise_pred, t):
        device = xt.device
        beta_t    = self.betas[t].to(device)
        alpha_t   = self.alphas[t].to(device)
        sqrt_acp  = self.sqrt_alpha_cum_prod[t].to(device)
        sqrt_1macp = self.sqrt_one_minus_alpha_cum_prod[t].to(device)

        x0 = (xt - sqrt_1macp * noise_pred) / sqrt_acp
        x0 = torch.clamp(x0, -1., 1.)

        mean = (xt - (beta_t / sqrt_1macp) * noise_pred) / torch.sqrt(alpha_t)

        if t == 0:
            return mean, x0

        alpha_cum_prod_prev = self.alpha_cum_prod[t - 1].to(device)
        variance = ((1 - alpha_cum_prod_prev) / (1 - self.alpha_cum_prod[t].to(device))) * beta_t
        sigma = variance ** 0.5
        return mean + sigma * torch.randn_like(xt), x0


scheduler = LinearNoiseScheduler(T, beta_start, beta_end)
print(f"ᾱ_T = {scheduler.alpha_cum_prod[-1]:.4f}  (should be ~0.006, not 0)")

## Model

In [ ]:
def get_time_embedding(time_steps, t_emb_dim):
    factor = 10000 ** (torch.arange(0, t_emb_dim // 2, device=time_steps.device) / (t_emb_dim // 2))
    t_emb  = time_steps[:, None].repeat(1, t_emb_dim // 2) / factor
    return torch.cat([torch.sin(t_emb), torch.cos(t_emb)], dim=-1)

In [ ]:
class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, down_sample, num_heads):
        super().__init__()
        self.down_sample = down_sample
        self.resnet_conv_first  = nn.Sequential(nn.GroupNorm(8, in_channels),  nn.SiLU(), nn.Conv2d(in_channels,  out_channels, 3, 1, 1))
        self.t_emb_layers       = nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, out_channels))
        self.resnet_conv_second = nn.Sequential(nn.GroupNorm(8, out_channels), nn.SiLU(), nn.Conv2d(out_channels, out_channels, 3, 1, 1))
        self.attention_norm     = nn.GroupNorm(8, out_channels)
        self.attention          = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.Conv2d(in_channels, out_channels, 1)
        self.down_sample_conv   = nn.Conv2d(out_channels, out_channels, 4, 2, 1) if down_sample else nn.Identity()

    def forward(self, x, t_emb):
        out = self.resnet_conv_first(x)
        out = out + self.t_emb_layers(t_emb)[:, :, None, None]
        out = self.resnet_conv_second(out) + self.residual_input_conv(x)
        B, C, H, W = out.shape
        attn_in = self.attention_norm(out.reshape(B, C, H*W)).transpose(1, 2)
        out_attn, _ = self.attention(attn_in, attn_in, attn_in)
        out = out + out_attn.transpose(1, 2).reshape(B, C, H, W)
        return self.down_sample_conv(out), out  


class MidBlock(nn.Module):
    def __init__(self, in_channels, mid_channels, out_channels, t_emb_dim, num_heads):
        super().__init__()
        self.res1_first  = nn.Sequential(nn.GroupNorm(8, in_channels),  nn.SiLU(), nn.Conv2d(in_channels,  mid_channels, 3, 1, 1))
        self.res1_second = nn.Sequential(nn.GroupNorm(8, mid_channels), nn.SiLU(), nn.Conv2d(mid_channels, mid_channels, 3, 1, 1))
        self.res2_first  = nn.Sequential(nn.GroupNorm(8, mid_channels), nn.SiLU(), nn.Conv2d(mid_channels, out_channels, 3, 1, 1))
        self.res2_second = nn.Sequential(nn.GroupNorm(8, out_channels), nn.SiLU(), nn.Conv2d(out_channels, out_channels, 3, 1, 1))
        self.t_emb1 = nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, mid_channels))
        self.t_emb2 = nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, out_channels))
        self.attn_norm = nn.GroupNorm(8, mid_channels)
        self.attention = nn.MultiheadAttention(mid_channels, num_heads, batch_first=True)
        self.skip1 = nn.Conv2d(in_channels,  mid_channels, 1)
        self.skip2 = nn.Conv2d(mid_channels, out_channels, 1)

    def forward(self, x, t_emb):
        res = x
        out = self.res1_first(x) + self.t_emb1(t_emb)[:, :, None, None]
        out = self.res1_second(out) + self.skip1(res)
        # Attention
        B, C, H, W = out.shape
        attn_in = self.attn_norm(out.reshape(B, C, H*W)).transpose(1, 2)
        out_attn, _ = self.attention(attn_in, attn_in, attn_in)
        out = out + out_attn.transpose(1, 2).reshape(B, C, H, W)
        res = out
        out = self.res2_first(out) + self.t_emb2(t_emb)[:, :, None, None]
        out = self.res2_second(out) + self.skip2(res)
        return out


class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, up_sample, num_heads):
        super().__init__()
        self.up_sample = up_sample
        self.resnet_conv_first  = nn.Sequential(nn.GroupNorm(8, in_channels),  nn.SiLU(), nn.Conv2d(in_channels,  out_channels, 3, 1, 1))
        self.t_emb_layers       = nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, out_channels))
        self.resnet_conv_second = nn.Sequential(nn.GroupNorm(8, out_channels), nn.SiLU(), nn.Conv2d(out_channels, out_channels, 3, 1, 1))
        self.attention_norm     = nn.GroupNorm(8, out_channels)
        self.attention          = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.Conv2d(in_channels, out_channels, 1)
        self.up_sample_conv     = nn.ConvTranspose2d(out_channels, out_channels, 4, 2, 1) if up_sample else nn.Identity()

    def forward(self, x, skip, t_emb):
        x   = torch.cat([x, skip], dim=1)
        out = self.resnet_conv_first(x)
        out = out + self.t_emb_layers(t_emb)[:, :, None, None]
        out = self.resnet_conv_second(out) + self.residual_input_conv(x)
        B, C, H, W = out.shape
        attn_in = self.attention_norm(out.reshape(B, C, H*W)).transpose(1, 2)
        out_attn, _ = self.attention(attn_in, attn_in, attn_in)
        out = out + out_attn.transpose(1, 2).reshape(B, C, H, W)
        return self.up_sample_conv(out)

In [ ]:
class Unet(nn.Module):
    def __init__(self, im_channels):
        super().__init__()
        self.down_channels = [32, 64, 128, 256]
        self.mid_channels  = [256, 256, 128]
        self.t_emb_dim     = 128

        self.t_proj = nn.Sequential(
            nn.Linear(self.t_emb_dim, self.t_emb_dim), nn.SiLU(),
            nn.Linear(self.t_emb_dim, self.t_emb_dim)
        )
        self.init_conv = nn.Conv2d(im_channels, self.down_channels[0], 3, padding=1)

        self.down_blocks = nn.ModuleList()
        for i in range(len(self.down_channels) - 1):
            down_sample = (i < len(self.down_channels) - 2)
            self.down_blocks.append(DownBlock(
                self.down_channels[i], self.down_channels[i+1],
                self.t_emb_dim, down_sample, num_heads=4))

        self.mid_block = MidBlock(*self.mid_channels, self.t_emb_dim, num_heads=4)

        self.up_blocks = nn.ModuleList()
        current_ch = self.mid_channels[-1]
        for i in reversed(range(len(self.down_channels) - 1)):
            skip_ch    = self.down_channels[i+1]
            out_ch     = self.down_channels[i]
            up_sample  = (i > 0)
            self.up_blocks.append(UpBlock(
                current_ch + skip_ch, out_ch,
                self.t_emb_dim, up_sample, num_heads=4))
            current_ch = out_ch

        self.final_conv = nn.Sequential(
            nn.GroupNorm(8, self.down_channels[0]), nn.SiLU(),
            nn.Conv2d(self.down_channels[0], im_channels, 3, padding=1)
        )

    def forward(self, x, t):
        t_emb = self.t_proj(get_time_embedding(t, self.t_emb_dim))
        out   = self.init_conv(x)
        skips = []
        for block in self.down_blocks:
            out, skip = block(out, t_emb)
            skips.append(skip)
        out = self.mid_block(out, t_emb)
        for block in self.up_blocks:
            out = block(out, skips.pop(), t_emb)
        return self.final_conv(out)


model = Unet(im_channels=im_channels).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

## Dataset

In [ ]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(IM_SIZE),
    torchvision.transforms.CenterCrop(IM_SIZE),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

full_dataset = torchvision.datasets.ImageFolder(root='./data/celeba', transform=transform)
indices      = random.sample(range(len(full_dataset)), min(num_samples, len(full_dataset)))
dataset      = Subset(full_dataset, indices)
dataloader   = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)

print(f"Dataset : {len(dataset):,} images")
print(f"Batches : {len(dataloader)} per epoch")

## Training

In [ ]:
optimizer    = torch.optim.Adam(model.parameters(), lr=lr)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
criterion    = nn.MSELoss()

def train(model, scheduler, dataloader, optimizer, lr_scheduler, criterion, epochs, T, device):
    model.train()
    all_losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        for x, _ in pbar:
            x         = x.to(device)
            t         = torch.randint(0, T, (x.shape[0],), device=device)
            noise     = torch.randn_like(x)
            noisy_x   = scheduler.add_noise(x, noise, t)
            noise_pred = model(noisy_x, t)
            loss = criterion(noise_pred, noise)

            optimizer.zero_grad()        
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        lr_scheduler.step()
        avg = epoch_loss / len(dataloader)
        all_losses.append(avg)
        print(f"Epoch {epoch+1}/{epochs} | loss={avg:.4f} | lr={optimizer.param_groups[0]['lr']:.6f}")

        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), f"ckpt_epoch{epoch+1}.pt")
            print(f"checkpoint saved")

    return all_losses

losses = train(model, scheduler, dataloader, optimizer, lr_scheduler, criterion, epochs, T, device)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(losses)+1), losses, marker='o')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('training_loss.png')
plt.show()

## Generation

In [ ]:
def generate_grid_gif(model, scheduler, T, device, save_path,
                      num_images=25, grid_nrow=5,
                      capture_every=10, fps=25, final_hold_secs=3):
    model.eval()
    frames, durations = [], []
    frame_ms = int(1000 / fps)
    hold_ms  = int(final_hold_secs * 1000)

    def to_grid_pil(x_batch):
        x_norm = torch.clamp((x_batch + 1.0) / 2.0, 0.0, 1.0)
        grid   = make_grid(x_norm, nrow=grid_nrow, padding=2, pad_value=1.0)
        arr    = grid.mul(255).add_(0.5).clamp_(0, 255).permute(1,2,0).cpu().to(torch.uint8).numpy()
        return Image.fromarray(arr)

    print(f"Sampling {num_images} images …")
    with torch.no_grad():
        x = torch.randn(num_images, 3, IM_SIZE, IM_SIZE).to(device)

        for i in tqdm(reversed(range(T)), total=T, desc="Denoising"):
            t_vec      = torch.full((num_images,), i, device=device, dtype=torch.long)
            noise_pred = model(x, t_vec)
            x, _       = scheduler.sample_prev_timestep(x, noise_pred, i)  # scalar i ✓

            if i % capture_every == 0 and i != 0:
                frames.append(to_grid_pil(x))
                durations.append(frame_ms)

        frames.append(to_grid_pil(x))
        durations.append(hold_ms)

    frames[0].save(
        save_path,
        save_all=True,
        append_images=frames[1:],
        duration=durations,   
        loop=0
    )
    total_s = sum(durations) / 1000
    print(f"Saved '{save_path}'  ({len(frames)} frames, {total_s:.1f}s total)")


generate_grid_gif(model, scheduler, T, device, "generation.gif",
                  num_images=25, grid_nrow=5,
                  capture_every=10, fps=25, final_hold_secs=3)